## Note on tokenizer vocabulary size (25,000)

The LSTM uses a `Tokenizer(num_words=25000)` (an integer-token sequence model), which is
intentionally **larger than the TF-IDF `max_features=5,000` used by the SVM, RF and XGBoost
baselines**. The two numbers serve different roles:

- TF-IDF features are a fixed-width sparse vector per document; pruning rare terms to the
  top-5k features keeps the linear classifiers tractable and acts as a regularizer.
- The LSTM operates on **integer token indices** mapped through a learned embedding layer
  (output dim 100). A wider vocabulary lets the embedding capture more clinical /
  Kinyarwanda-derived rare terms that would otherwise collapse to `<OOV>`.

Tokens beyond rank 25,000 are mapped to `<OOV>`. After tokenization, sequences are padded
or truncated to `max_len=200`.


In [1]:
!pip install scikit-multilearn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.4/89.4 kB 10.2 MB/s eta 0:00:00


In [2]:
import pandas as pd
import random
import numpy as np
import torch
def set_seed(seed):
    #Set seed for reproducibility
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

In [3]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

In [4]:
import pandas as pd
import torch
from skmultilearn.model_selection import iterative_train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer

In [5]:
dfCovid = pd.read_csv('data/dataRwaCovid.csv')

In [6]:
dfDiabetes = pd.read_csv('data/dataIHADiabetes.csv')

In [7]:
DiabetesTopics = [
    "Physical",
    "Psychological",
    "No Symptoms",
    "Prognosis",
    "Laboratory/Testing",
    "Imaging",
    "Clinical",
    "Testing/Monitoring Devices",
    "Health Data",
    "Diagnostic Methods - Other",
    "Medications",
    "Procedures",
    "Alternative",
    "Physical Therapy",
    "Counseling",
    "Adverse Events",
    "Therapeutic Devices",
    "Treatment(Rx) - Other",
    "Outpatient Logistics/Scheduling",
    "Hospitalizations",
    "Insurance/Billing",
    "Medical Records",
    "Referrals",
    "Transportation",
    "Primary (Pharmaceutical Prevention)",
    "Primary (Non-Pharmaceutical Prevention)",
    "Secondary (Pharmaceutical Prevention)",
    "Secondary (Non-Pharmaceutical Prevention)",
    "Diet/Nutrition",
    "Exercise",
    "Substance Use",
    "Entertainment",
    "Lifestyle - Other",
    "Housing",
    "Work/School",
    "Social Services",
    "Friends & Family",
    "Cultural/Religion",
    "Travel",
    "Physical Environment/Climate",
    "Financial",
    "Social - Other",
    "Technical/IT",
    "Safety Concerns",
    "Health Education",
    "Sexual & Reproductive Health",
    "Child & Family Health",
    "Problems Solved",
    "Grateful Patient",
    "Service Complaint",
    "Request to Stop",
    "Emergent",
    "Urgent",
    "Non-urgent",
    "Stigma Present",
    "Rapport",
    "Transition to Adult Clinic"
]

In [8]:
CovidTopics = [
    "Physical",
    "Mental/Emotional",
    "No Symptoms",
    "Laboratory/Testing",
    "Imaging",
    "Clinical",
    "Diagnostic Methods - Other",
    "Medications",
    "Procedures",
    "Alternative",
    "Physical Therapy",
    "Counseling",
    "Treatment(Rx) - Other",
    "Outpatient Logistics/Scheduling",
    "Hospitalizations",
    "Pharmaceutical Prevention",
    "Non-Pharmaceutical Prevention",
    "Diet/Nutrition",
    "Exercise",
    "Substance Use",
    "Lifestyle - Other",
    "Housing",
    "Work/School",
    "Social Services",
    "Friends & Family",
    "Cultural/Religion",
    "Travel",
    "Physical Environment/Climate",
    "Financial",
    "Social - Other",
    "Technical/IT",
    "Safety concern",
    "Health Education",
    "Maternal & Child Health",
    "Problems Solved",
    "Grateful Patient",
    "Service Complaint",
    "Request to Stop",
    "Emergent",
    "Urgent",
    "Non-urgent",
    "Stigma Present",
    "wave",
    "batch"
]

In [9]:
commonSubtopics = sorted(list(set(DiabetesTopics) & set(CovidTopics)))

In [10]:
dfCovid[commonSubtopics].sum()

,0
Alternative,35
Clinical,5
Counseling,0
Cultural/Religion,154
Diagnostic Methods - Other,5
Diet/Nutrition,102
Emergent,138
Exercise,25
Financial,53
Friends & Family,138


In [11]:
diabetesColumns = commonSubtopics[:]
diabetesColumns.insert(0,"conversation")
dfDiabetesNew = dfDiabetes[diabetesColumns]


In [12]:
covidColumns = commonSubtopics[:]
covidColumns.insert(0,"conversation(english_only)")
dfCovidNew = dfCovid[covidColumns]


In [13]:
dfCovidNew = dfCovidNew.rename(columns={"conversation(english_only)":"conversation"})

In [14]:
dfCombined = pd.concat([dfCovidNew, dfDiabetesNew])

In [15]:
dfCombined.sum()

,0
conversation,"(S): Hello, how are you today? Do you have any..."
Alternative,35
Clinical,6
Counseling,0
Cultural/Religion,343
Diagnostic Methods - Other,8
Diet/Nutrition,340
Emergent,140
Exercise,131
Financial,53


In [16]:
n = 100
label_sum = dfCombined[commonSubtopics].sum()
label_keep = label_sum[label_sum >= n].index.tolist()
label_keep_original = label_keep[:]
label_keep.insert(0,"conversation")
dfCombined_filtered = dfCombined[label_keep]



In [17]:
dfCombined_filtered.sum()

,0
conversation,"(S): Hello, how are you today? Do you have any..."
Cultural/Religion,343
Diet/Nutrition,340
Emergent,140
Exercise,131
Friends & Family,291
Grateful Patient,291
Health Education,322
Laboratory/Testing,1078
Medications,418


In [18]:
x = dfCombined_filtered["conversation"].to_numpy()
y = dfCombined_filtered[label_keep_original].to_numpy()

In [19]:
df_aug = pd.read_csv('data/backtranslated_train.csv')

In [20]:
import numpy as np

split = np.load("data/shared_split_indices.npz")
train_idx = split["train_idx"]
val_idx   = split["val_idx"]
test_idx  = split["test_idx"]

x_raw = dfCombined_filtered["conversation"].to_numpy()
y     = dfCombined_filtered[label_keep_original].to_numpy()

x_val  = x_raw[val_idx].reshape(-1, 1)
x_test = x_raw[test_idx].reshape(-1, 1)
y_val  = y[val_idx]
y_test = y[test_idx]

df_aug_train = pd.read_csv("data/backtranslated_train.csv")
x_train = df_aug_train["conversation"].to_numpy()
y_train = df_aug_train[label_keep_original].to_numpy()

In [21]:
x_train.shape
type(x_train)


numpy.ndarray

In [22]:
y_train.shape
type(y_train)

numpy.ndarray

In [23]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [24]:
tokenizer = Tokenizer(num_words=25000, oov_token="<OOV>")
tokenizer.fit_on_texts(x_train.flatten().tolist())

In [25]:
def padAndTokenize(text,tokenizer,max_length):
  sequences = tokenizer.texts_to_sequences(text)
  paddes_seq = pad_sequences(sequences,maxlen=max_length,padding='post',truncating='post')
  return paddes_seq

In [26]:
x_train_tokAndPadded = padAndTokenize(x_train.flatten(),tokenizer,512)
x_val_tokAndPadded = padAndTokenize(x_val.flatten(),tokenizer,512)
x_test_tokAndPadded = padAndTokenize(x_test.flatten(),tokenizer,512)

word_index_dic = tokenizer.word_index
vocab_len = len(word_index_dic) + 1
print(x_test_tokAndPadded)

[[  4  19  15 ...   0   0   0]
 [  4  19  15 ...   0   0   0]
 [  4 186   3 ...   0   0   0]
 ...
 [  4 262 683 ...   0   0   0]
 [  4 146 285 ...   0   0   0]
 [  4 345 130 ...   0   0   0]]


In [27]:
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=vocab_len, output_dim=128, input_length=512),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(128, return_sequences=True)),
    tf.keras.layers.GlobalMaxPooling1D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(y_train.shape[1], activation='sigmoid')
])

In [28]:
model.compile(loss='binary_crossentropy', optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),metrics=[tf.keras.metrics.F1Score(average='macro',threshold=0.5)])

In [29]:
history = model.fit(x_train_tokAndPadded, y_train.astype('float32'),
                    epochs=350,
                    validation_data=(x_val_tokAndPadded, y_val.astype('float32')),
                    batch_size=64)

Epoch 1/350
86/86 [==============================] - 19s 162ms/step - loss: 0.5737 - f1_score: 0.1611 - val_loss: 0.3143 - val_f1_score: 0.0495
Epoch 2/350
86/86 [==============================] - 9s 105ms/step - loss: 0.3491 - f1_score: 0.0997 - val_loss: 0.2948 - val_f1_score: 0.0495
Epoch 3/350
86/86 [==============================] - 8s 98ms/step - loss: 0.3340 - f1_score: 0.0865 - val_loss: 0.2880 - val_f1_score: 0.0495
Epoch 4/350
86/86 [==============================] - 6s 68ms/step - loss: 0.3184 - f1_score: 0.1038 - val_loss: 0.2745 - val_f1_score: 0.0878
Epoch 5/350
86/86 [==============================] - 6s 71ms/step - loss: 0.3036 - f1_score: 0.1106 - val_loss: 0.2732 - val_f1_score: 0.0865
Epoch 6/350
86/86 [==============================] - 6s 71ms/step - loss: 0.3042 - f1_score: 0.1006 - val_loss: 0.2825 - val_f1_score: 0.0495
Epoch 7/350
86/86 [==============================] - 5s 56ms/step - loss: 0.3083 - f1_score: 0.0929 - val_loss: 0.2810 - val_f1_score: 0.0495
Epo

In [30]:
from sklearn.metrics import f1_score, classification_report
import numpy as np

y_pred_proba = model.predict(x_test_tokAndPadded)
y_pred = (y_pred_proba > 0.5).astype(int)

test_loss, test_f1 = model.evaluate(x_test_tokAndPadded, y_test.astype('float32'))
print(f"Test Loss: {test_loss:.4f}")

print("\n--- Per-Class F1 Scores ---")
for i, label_name in enumerate(label_keep_original):
    f1 = f1_score(y_test[:, i], y_pred[:, i])
    print(f"Test F1 Score for {label_name}: {f1:.4f}")

macro_f1 = f1_score(y_test, y_pred, average='macro')
print(f"\nMacro F1 Score: {macro_f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=label_keep_original, zero_division=0))

19/19 [==============================] - 0s 15ms/step - loss: 0.3335 - f1_score: 0.5806
Test Loss: 0.3335

--- Per-Class F1 Scores ---
Test F1 Score for Cultural/Religion: 0.7921
Test F1 Score for Diet/Nutrition: 0.6591
Test F1 Score for Emergent: 0.2051
Test F1 Score for Exercise: 0.5806
Test F1 Score for Friends & Family: 0.4167
Test F1 Score for Grateful Patient: 0.4615
Test F1 Score for Health Education: 0.7407
Test F1 Score for Laboratory/Testing: 0.9357
Test F1 Score for Medications: 0.6240
Test F1 Score for No Symptoms: 0.8614
Test F1 Score for Non-urgent: 0.9243
Test F1 Score for Outpatient Logistics/Scheduling: 0.5915
Test F1 Score for Physical: 0.6629
Test F1 Score for Physical Environment/Climate: 0.8421
Test F1 Score for Service Complaint: 0.0000
Test F1 Score for Social Services: 0.6400
Test F1 Score for Technical/IT: 0.4444
Test F1 Score for Urgent: 0.2927
Test F1 Score for Work/School: 0.3571

Macro F1 Score: 0.5806

Classification Report:
                               

In [31]:

import numpy as np
from sklearn.metrics import f1_score

N_BOOTSTRAP = 1000
rng = np.random.default_rng(42)
n = len(y_test)

boot_scores = []
for _ in range(N_BOOTSTRAP):
    idx = rng.integers(0, n, size=n)
    score = f1_score(y_test[idx], y_pred[idx], average='macro', zero_division=0)
    boot_scores.append(score)

boot_scores = np.array(boot_scores)
lo = np.percentile(boot_scores, 2.5)
hi = np.percentile(boot_scores, 97.5)
print(f"Macro F1: {macro_f1:.4f}  95% CI [{lo:.4f}, {hi:.4f}]")

Macro F1: 0.5806  95% CI [0.5443, 0.6104]


In [32]:
import numpy as np, os
os.makedirs("predictions", exist_ok=True)


np.savez_compressed(
    "predictions/LSTM_preds.npz",
    y_true = y_test,
    y_pred = y_pred,
)
print("Saved predictions/LSTM_preds.npz")

Saved predictions/LSTM_preds.npz


## Per-corpus + per-label evaluation (LSTM)

Loads saved `LSTM_preds.npz` and reports Macro F1 (+ 95% bootstrap CI) and per-label F1 separately for the pooled corpus and for the Rwanda and Canada halves. Source labels come from `Ensemble_test_predictions.json`, which carries the per-row `source_corpus` field for the same 584-doc test set.


In [ ]:
# PERCORPUSCELL_v1
# Per-corpus + per-label evaluation against the shared test split, with 1,000-resample bootstrap CIs.
import json, numpy as np
from sklearn.metrics import f1_score, classification_report

with open("predictions/Ensemble_test_predictions.json") as f:
    _ens = json.load(f)
_src    = np.array([r["source_corpus"] for r in _ens])
_labels = list(_ens[0]["true_labels"].keys())

_d = np.load("predictions/LSTM_preds.npz")
_yt, _yp = _d["y_true"].astype(int), _d["y_pred"]
if _yp.dtype != int:
    _yp = (_yp > 0.5).astype(int) if _yp.max() <= 1 else _yp.astype(int)

for _corpus in ["Combined", "Rwanda", "Canada"]:
    _mask = np.ones(len(_src), bool) if _corpus == "Combined" else (_src == _corpus)
    _yti, _ypi = _yt[_mask], _yp[_mask]
    _macro = f1_score(_yti, _ypi, average="macro", zero_division=0)

    _rng = np.random.default_rng(42)
    _n = len(_yti)
    _boot = np.empty(1000)
    for _i in range(1000):
        _idx = _rng.integers(0, _n, _n)
        _boot[_i] = f1_score(_yti[_idx], _ypi[_idx], average="macro", zero_division=0)
    _lo, _hi = np.percentile(_boot, [2.5, 97.5])

    print(f"\n=== {{_corpus}} (n={{_n}})  Macro F1 {{_macro:.4f}}  95% CI [{{_lo:.4f}}, {{_hi:.4f}}] ===")
    print(classification_report(_yti, _ypi, target_names=_labels, zero_division=0))
